In [ ]:
import pandas as pd
import numpy as np

# Load the dataset
# Note: if your file is in the 'data' folder, use the relative path
df = pd.read_excel('../data/online_retail_II.xlsx')

# Show the first 5 rows to confirm it loaded
df.head()

In [ ]:
# Check for missing values in each column
print(df.isnull().sum())

# Drop rows where Customer ID is missing
df.dropna(subset=['Customer ID'], inplace=True)

# Verify they are gone
print(f"Remaining rows: {df.shape[0]}")

In [ ]:
# Remove rows where Invoice contains 'C'
df = df[~df['Invoice'].astype(str).str.contains('C')]

In [ ]:
# Keep only positive quantities and prices
df = df[(df['Quantity'] > 0) & (df['Price'] > 0)]

In [ ]:
# Calculate Total Sum
df['TotalSum'] = df['Quantity'] * df['Price']
df.head()

In [ ]:
# Calculate the difference
original_count = 1048575 # This varies based on your specific file version
current_count = df.shape[0]
removed_count = original_count - current_count

print(f"Original Row Count: {original_count}")
print(f"Cleaned Row Count: {current_count}")
print(f"Total Rows Removed: {removed_count}")
print(f"Percentage of Data Retained: {round((current_count / original_count) * 100, 2)}%")

In [ ]:
import datetime as dt

# Find the last date in the dataset and add one day
snapshot_date = df['InvoiceDate'].max() + dt.timedelta(days=1)
print(f"Analysis Snapshot Date: {snapshot_date}")

In [ ]:
rfm = df.groupby('Customer ID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,
    'Invoice': lambda x: x.nunique(),
    'TotalSum': 'sum'
})

# Rename the columns for clarity
rfm.columns = ['Recency', 'Frequency', 'Monetary']

rfm.head()

In [ ]:
rfm['Recency'] = rfm['Recency'].astype(int)

In [ ]:
# Create Recency scores (5 is best/most recent)
rfm["recency_score"] = pd.qcut(rfm['Recency'], 5, labels=[5, 4, 3, 2, 1])

# Create Frequency scores (5 is best/most frequent)
# Note: We use 'rank(method="first")' because e-commerce data often has many customers with the same frequency
rfm["frequency_score"] = pd.qcut(rfm['Frequency'].rank(method="first"), 5, labels=[1, 2, 3, 4, 5])

# Create Monetary scores (5 is best/highest spender)
rfm["monetary_score"] = pd.qcut(rfm['Monetary'], 5, labels=[1, 2, 3, 4, 5])

# Combine Recency and Frequency to create a segment map
# (Monetary is often used for secondary filtering)
rfm["RFM_SCORE"] = (rfm['recency_score'].astype(str) + 
                    rfm['frequency_score'].astype(str))

rfm.head()

In [ ]:
seg_map = {
    r'[1-2][1-2]': 'hibernating',
    r'[1-2][3-4]': 'at_Risk',
    r'[1-2]5': 'cant_loose',
    r'3[1-2]': 'about_to_sleep',
    r'33': 'need_attention',
    r'[3-4][4-5]': 'loyal_customers',
    r'41': 'promising',
    r'51': 'new_customers',
    r'[4-5][2-3]': 'potential_loyalists',
    r'5[4-5]': 'champions'
}

rfm['segment'] = rfm['RFM_SCORE'].replace(seg_map, regex=True)
rfm.head()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12, 8))
sns.countplot(data=rfm, y='segment', order=rfm['segment'].value_counts().index, palette='viridis')
plt.title('Customer Segmentation Distribution')
plt.show()